# Misión Asíncrona: Cazando al Falso Influencer
Su misión es extraer la red, construir el grafo y descubrir matemáticamente quién es el fraude.

In [1]:
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import pandas as pd

df_base = pd.read_csv('../datasets/export_api_influencers.csv')

df_base.head()

,Usuario,Ubicacion,Fecha_Creacion,Cuentas_Que_Sigue
0,Bot_5,India,2023-05-06,AMATEUR_1
1,fan_6,Mexico,2023-07-14,"GARNACHO, MESSI"
2,Bot_11,India,2023-02-14,AMATEUR_1
3,Bot_1,India,2023-11-24,AMATEUR_1
4,Bot_36,India,2023-06-24,AMATEUR_1


**Fase 1: Data Engineering (La Explosión)** Para poder usar networkx, primero necesitan una Lista de Aristas purificada. Abran un nuevo Jupyter Notebook, importen pandas y realicen la siguiente tubería de datos (Data Pipeline):

- Limpieza Estructural: Eliminen las filas donde la columna Cuentas_Que_Sigue esté vacía (nula). Estandaricen la columna Usuario a MAYÚSCULAS y quítenle los espacios al inicio/final. Eliminen usuarios duplicados.
- La Explosión: Separen los nombres de la columna Cuentas_Que_Sigue usando .str.split(','). Luego, utilicen la función mágica de Pandas .explode() sobre esa columna para aplanar la tabla y convertirla en una verdadera lista de Origen -> Destino.
- Purificación Post-Explosión: Vuelvan a limpiar los textos de la columna destino (mayúsculas y sin espacios extra). Finalmente, eliminen los Self-Loops (Registros donde el Usuario aparezca siguiéndose a sí mismo).

In [2]:
# Limpieza estructural
df = df_base.dropna(subset=["Cuentas_Que_Sigue"]).copy()
df["Usuario"] = df["Usuario"].str.upper().str.strip()
df = df.drop_duplicates(subset="Usuario")

# Explosión
df["Cuentas_Que_Sigue"] = df["Cuentas_Que_Sigue"].str.split(',')
df_aristas = df[["Usuario", "Cuentas_Que_Sigue"]].explode("Cuentas_Que_Sigue")
df_aristas = df_aristas.rename(
    columns={"Usuario": "Origen", "Cuentas_Que_Sigue": "Destino"}
)

# Purificación
df_aristas["Destino"] = df_aristas["Destino"].str.upper().str.strip()
df_aristas = df_aristas[df_aristas["Destino"] != ""]
df_aristas = df_aristas[df_aristas["Origen"] != df_aristas["Destino"]]

df_aristas.head()

,Origen,Destino
0,BOT_5,AMATEUR_1
1,FAN_6,GARNACHO
1,FAN_6,MESSI
2,BOT_11,AMATEUR_1
3,BOT_1,AMATEUR_1


**Fase 2: El Engaño de la Popularidad Con su Lista de Aristas terminada:**
- Importen networkx y conviertan su tabla en un Grafo Dirigido (nx.DiGraph).
- Calculen la Centralidad de Entrada (nx.in_degree_centrality).
- Impriman a los 3 usuarios con el puntaje más alto.
- Pregunta: A simple vista, ¿quién parece ser el rey de la red?

In [3]:
# Creación del grafo
G = nx.from_pandas_edgelist(
    df_aristas, source="Origen", target="Destino", create_using=nx.DiGraph()
)

centralidad = nx.in_degree_centrality(G)
top3 = sorted(centralidad.items(), key=lambda x: x[1], reverse=True)[:3]

for usuario, puntaje in top3:
    print(f"{usuario}: {puntaje:.4f}")

AMATEUR_1: 0.7231
GARNACHO: 0.2000
MESSI: 0.1538


**Fase 3: La Verdad Topológica (Las Métricas)** Es hora de usar los algoritmos que vimos en los videos de hoy para auditar la red.
- Calculen el PageRank (nx.pagerank). ¿Quién tiene ahora la verdadera autoridad?
- Conviertan el grafo a No Dirigido y calculen el Coeficiente de Clustering (nx.clustering). Impriman el valor exacto de AMATEUR_1 y el de GARNACHO.

In [4]:
# PageRank
pr = nx.pagerank(G)
top_pr = sorted(pr.items(), key=lambda x: x[1], reverse=True)[:3]
for usuario, puntaje in top_pr:
    print(f"PageRank {usuario}: {puntaje:.4f}")

# Clustering
G_und = G.to_undirected()
print("Clustering AMATEUR_1:", nx.clustering(G_und, "AMATEUR_1"))
print("Clustering GARNACHO:", nx.clustering(G_und, "GARNACHO"))

PageRank AMATEUR_1: 0.1969
PageRank DE_PAUL: 0.0883
PageRank GARNACHO: 0.0843
Clustering AMATEUR_1: 0
Clustering GARNACHO: 0.13333333333333333


**Veredicto Final (Markdown):** Cierren su notebook con una celda de texto. Usando estrictamente sus resultados matemáticos de PageRank y Clustering, argumenten de forma técnica y profesional:
- **¿Quién compró bots y por qué?**
Los bots los compró AMATEUR_1 ya que tiene alta centralidad pero su coeficiente de clustering es de 0. Esto quiere decir que ninguno de sus seguidores está conectado con otro, lo cual no es como una comunidad orgánica debería comportarse. 

- **¿Quién es el influencer real y por qué?**
GARNACHO, ya que aunque tiene menos seguidores que AMATEUR_1 pero mantiene un PageRank competitivo y tiene un clustering mayor a 0. Esto nos dice que sus seguidores interactúan entre sí, por lo que no son bots.